# Search-09b : Minima fallacieux — le paysage de la relaxation SDP factorisée

**Navigation** : [Version C#](Search-09b-SpuriousMinima-CSharp.ipynb) | [<< Search-2c (QuikGraph)](Search-02c-QuikGraph.ipynb) | [↑ Série Search](../README.md) | [Partie 2 : CSP →](../Part2-CSP/README.md)

**Question centrale.** Search-9 a relaxé un problème combinatoire vers un programme linéaire — relaxation *convexe*, résolue numériquement. Ici on fait un pas de plus, et un pas de côté : la relaxation **semidéfinie** de MaxCut est convexe, mais les solveurs à grande échelle ne la résolvent pas telle quelle — ils la **factorisent** $Y = XX^T$ en une variable $X \in \mathbb{R}^{n \times r}$ de rang borné, et le problème devient **non convexe**. La question de ce notebook : *quand ce paysage non convexe piège-t-il la recherche locale dans des minima fallacieux* — des points stationnaires sous-optimaux — et quand est-il prouvablement sûr ?

C'est le point d'arrivée « certification » du fil paysages : MGS-15 (série .NET de la Partie 4) *mesurait* les paysages de fonctions par corrélation fitness-distance ; ici on interroge un paysage dont la théorie étudie sous des hypothèses de généricité et de régularité. Pilote « calculatoire borné » de l'EPIC #13106 (digestion et canonicalisation des mathématiques assistées) : chaque affirmation est mesurée sur des instances exhaustivement énumérables, avec référence numérique par vrai solveur conique.

## A. Le problème et sa relaxation

Soit $W$ une matrice de poids symétrique à diagonale nulle (un graphe pondéré). MaxCut cherche la bipartition $s \in \{\pm 1\}^n$ maximisant

$$\mathrm{cut}(s) = \frac{1}{2} \sum_{i<j} W_{ij}\,(1 - s_i s_j).$$

La relaxation de Goemans–Williamson remplace chaque $s_i s_j$ par un produit scalaire $\langle y_i, y_j \rangle$ de vecteurs unité — c'est-à-dire $Y = XX^T$ avec $\mathrm{diag}(Y) = 1$ — et lève l'exigence de rang :

$$\text{(SDP)} \quad \max_Y \ \tfrac{1}{2}\sum_{i<j} W_{ij}(1 - Y_{ij}) \quad \text{s.c.} \quad \mathrm{diag}(Y)=1, \ Y \succeq 0.$$

Le problème (SDP) est convexe : sa valeur $p^*$ est estimée numériquement (cône semidéfini, solveur conique). C'est la référence de tout ce qui suit.

In [1]:
import itertools
import numpy as np
import cvxpy as cp

def instance_cycle6():
    n = 6
    W = np.zeros((n, n))
    for i in range(n):
        W[i, (i + 1) % n] = W[(i + 1) % n, i] = 1.0
    return "C6 (cycle pair, biparti)", W

def instance_k8():
    n = 8
    rng = np.random.default_rng(42)
    Wr = rng.uniform(0.1, 1.0, (n, n))
    W = np.triu(Wr, 1) + np.triu(Wr, 1).T
    return "K8 poids uniformes seed 42", W

def instance_g10():
    n = 10
    rng = np.random.default_rng(7)
    A = (rng.random((n, n)) < 0.5).astype(float)
    W = np.triu(A, 1) + np.triu(A, 1).T
    return "G(10, 0.5) seed 7", W

INSTANCES = [instance_cycle6(), instance_k8(), instance_g10()]
for name, W in INSTANCES:
    aretes = int(np.count_nonzero(np.triu(W)))
    print(f"{name:32s} n={len(W):2d}  aretes={aretes:2d}  poids total={np.triu(W).sum():7.3f}")

C6 (cycle pair, biparti)         n= 6  aretes= 6  poids total=  6.000
K8 poids uniformes seed 42       n= 8  aretes=28  poids total= 18.130
G(10, 0.5) seed 7                n=10  aretes=19  poids total= 19.000


In [2]:
def maxcut_brut(W):
    n = len(W)
    meilleur = -np.inf
    for bits in itertools.product([-1, 1], repeat=n - 1):
        s = np.array([1] + list(bits))
        v = 0.5 * sum(W[i, j] * (1 - s[i] * s[j])
                      for i in range(n) for j in range(i + 1, n))
        if v > meilleur:
            meilleur = v
    return meilleur

def sdp_maxcut(W):
    n = len(W)
    Y = cp.Variable((n, n), PSD=True)
    obj = cp.Maximize(0.25 * cp.sum(cp.multiply(W, 1 - Y)))
    prob = cp.Problem(obj, [cp.diag(Y) == 1])
    prob.solve(solver=cp.CLARABEL)
    return prob.value

print(f"{'instance':32s} {'cut optimal (brute)':>20s} {'SDP (CLARABEL)':>15s} {'ecart':>8s}")
for name, W in INSTANCES:
    b, s = maxcut_brut(W), sdp_maxcut(W)
    print(f"{name:32s} {b:20.4f} {s:15.4f} {s - b:8.4f}")

instance                          cut optimal (brute)  SDP (CLARABEL)    ecart
C6 (cycle pair, biparti)                       6.0000          6.0000  -0.0000
K8 poids uniformes seed 42                    11.8581         11.8584   0.0003
G(10, 0.5) seed 7                             14.0000         14.3249   0.3249


**Lecture.** Sur le cycle pair $C_6$, la relaxation est *exacte* : $\mathrm{SDP} = \mathrm{cut} = 6$ (un graphe biparti a une coupe qui traverse toutes les arêtes, et $Y = ss^T$ l'atteint). Sur $K_8$ pondéré les valeurs numériques sont proches, avec un écart de $3\times10^{-4}$. Sur $G(10,\,0.5)$ en revanche l'écart vaut $0{,}3249$ : la valeur semidéfinie $14{,}3249$ est **inatteignable par toute coupe entière** (la meilleure coupe vaut $14$). C'est l'écart intégralité de la relaxation — et la raison pour laquelle la suite distingue soigneusement *valeur SDP* (la cible du paysage factorisé) et *coupe* (ce qu'on en arrondit).

### Contrôle : le cycle impair rompt l'exactitude de la relaxation

La lecture ci-dessus attribue l'exactitude au fait que $C_6$ est **biparti** — « un graphe biparti a une coupe qui traverse toutes les arêtes ». La table, elle, nomme l'instance « cycle pair » : les deux formulations ne se distinguent que sur un cas, un cycle **impair**, de taille voisine et de même construction. Le contrôle ci-dessous ajoute $C_5$ et $C_7$ aux instances et recalcule les deux quantités avec le même code.

**Ce qui aurait affaibli la lecture** : un écart $\mathrm{SDP} - \mathrm{cut}$ resté nul sur $C_5$ et sur $C_7$. L'exactitude ne serait alors pas une propriété de la bipartition, et l'énoncé du carnet serait incomplet dans l'autre sens. Le résultat attendu par la théorie — un écart strictement positif sur les cycles impairs — ne compte que s'il est mesuré : il est calculé ici par énumération exhaustive exacte, en face du même solveur conique.

In [3]:
# Controle : le cycle impair rompt l'exactitude de la relaxation
# Meme construction, meme enumeration exhaustive, meme solveur conique que la table precedente.

def instance_cycle(n):
    """Cycle non pondere a n sommets, construit comme instance_cycle6."""
    W = np.zeros((n, n))
    for i in range(n):
        W[i, (i + 1) % n] = W[(i + 1) % n, i] = 1.0
    return W


CYCLES = [(f"C{n}", instance_cycle(n)) for n in (5, 6, 7)]

print(f"{'cycle':6s} {'aretes':>7s} {'biparti':>8s} {'cut optimal':>12s} "
      f"{'SDP':>9s} {'ecart':>9s} {'ecart/arete':>12s}")
print("-" * 70)
for name, W in CYCLES:
    n = len(W)
    aretes = int(np.count_nonzero(np.triu(W)))
    coupe, sdp = maxcut_brut(W), sdp_maxcut(W)
    print(f"{name:6s} {aretes:7d} {str(n % 2 == 0):>8s} {coupe:12.4f} "
          f"{sdp:9.4f} {sdp - coupe:9.4f} {(sdp - coupe) / aretes:12.4f}")

cycle   aretes  biparti  cut optimal       SDP     ecart  ecart/arete
----------------------------------------------------------------------


C5           5    False       4.0000    4.5225    0.5225       0.1045
C6           6     True       6.0000    6.0000   -0.0000      -0.0000


C7           7    False       6.0000    6.6534    0.6534       0.0933


### Lecture du résultat : l'écart suit la bipartition, pas la taille du cycle

**Sortie obtenue** : la coupe optimale vaut 4 sur $C_5$, 6 sur $C_6$ et 6 sur $C_7$ ; la valeur semidéfinie vaut 4.5225, 6.0000 et 6.6534. L'écart $\mathrm{SDP} - \mathrm{cut}$ vaut 0.5225 sur le cycle impair à cinq sommets, -0.0000 sur $C_6$ — nul à la tolérance du solveur — et 0.6534 sur $C_7$.

**Ce que le contrôle établit.** Les trois cycles mesurés sont de tailles voisines (cinq, six et sept sommets), donc le résultat se lit comme une comparaison de trois cycles, non comme un contrôle à taille fixée. Sur ces trois instances, l'écart ne suit pas la taille : la relaxation est strictement supérieure à toute coupe entière sur $C_5$ et sur $C_7$, alors qu'elle est exacte sur $C_6$. Il suit la bipartition : sur un graphe biparti, une coupe traverse toutes les arêtes, donc $\mathrm{cut} = |E|$, qui est aussi la borne supérieure de la relaxation, d'où l'égalité. Un cycle impair n'admet aucune coupe parfaite — une arête au moins reste intacte — et la relaxation y trouve mieux que n'importe quelle coupe entière. Dans cette famille, « pair » et « biparti » désignent le même fait : le contrôle ne sépare donc pas deux explications indépendantes, il montre que l'écart d'intégralité accompagne la bipartition sur les instances testées.

**Ce que cela change pour la suite.** L'écart d'intégralité n'est pas un accident de grande instance : il apparaît dès le plus petit des cycles impairs testés, et l'écart rapporté au nombre d'arêtes reste d'ordre 0.1 sur les deux tailles mesurées (0.1045 sur $C_5$, 0.0933 sur $C_7$) : il ne s'effondre pas quand la taille augmente, alors que l'écart total, lui, s'élargit (0.5225 puis 0.6534). La valeur semidéfinie reste donc une cible dont la coupe arrondie peut demeurer strictement en dessous sur ces instances, ce que la suite du carnet doit garder en tête quand elle compare les candidats du paysage factorisé à la valeur de référence.

## B. La factorisation de Burer–Monteiro

Résoudre (SDP) en variable matricielle $n \times n$ coûte cher en grande dimension : $O(n^2)$ inconnues et un cône de taille $n$. L'idée de Burer–Monteiro : paramétrer $Y = XX^T$ avec $X \in \mathbb{R}^{n \times r}$, $r \ll n$. La contrainte $Y \succeq 0$ devient automatique, la contrainte $\mathrm{diag}(Y)=1$ force chaque ligne $x_i$ sur la sphère $S^{r-1}$, et le problème devient

$$\text{(BM}_r\text{)} \quad \max_{X \in (S^{r-1})^n} \ f(X) = \tfrac{1}{2}\sum_{i<j} W_{ij}\,(1 - \langle x_i, x_j\rangle).$$

Non convexe en $X$ — et c'est *ainsi* que tournent les solveurs SDP à grande échelle. Le prix : des minima locaux fallacieux deviennent possibles. La théorie étudie ce prix sous des hypothèses précises :

- **Barvinok (1995) – Pataki (1998)** : un SDP à $m$ contraintes affines admet une solution optimale de rang $r$ avec $r(r+1)/2 \le m$. Le rang optimal n'est jamais « grand ».
- **Burer–Monteiro (2005)** (*Local minima and convergence in low-rank semidefinite programming*, Math. Program. 103(3), 427–444) : si $r(r+1)/2 > m$ et sous une condition de non-dégénérescence, des résultats de paysage sans piège sont établis sous des hypothèses techniques et de généricité qui ne sont pas vérifiées pour ces matrices.
- Pont moderne vers les sommes de carrés : *Low-Rank Univariate Sum of Squares Has No Spurious Local Minima* (SIAM J. Optim., 2023) montre la même phénoménologie pour les programmes SOS factorisés en rang borné.

Pour MaxCut, $m = n$ (les $n$ contraintes diagonales), d'où le seuil $r^* = \min\{r : r(r+1)/2 > n\}$. La prédiction falsifiable de ce notebook : comparer les candidats observés aux rangs sous et au-dessus de $r^*$, sans certifier ces hypothèses.

In [4]:
def r_star(m):
    r = 1
    while r * (r + 1) // 2 <= m:
        r += 1
    return r

print(f"{'instance':32s} {'n (= m)':>7s} {'r*':>3s}   justification du seuil")
for name, W in INSTANCES:
    n = len(W)
    rs = r_star(n)
    print(f"{name:32s} {n:7d} {rs:3d}   r={rs-1}: {r_star(n)-1}*{r_star(n)}/2 = {(rs-1)*rs//2} <= {n} ; "
          f"r={rs}: {rs}*{rs+1}/2 = {rs*(rs+1)//2} > {n}")

instance                         n (= m)  r*   justification du seuil
C6 (cycle pair, biparti)               6   4   r=3: 3*4/2 = 6 <= 6 ; r=4: 4*5/2 = 10 > 6
K8 poids uniformes seed 42             8   4   r=3: 3*4/2 = 6 <= 8 ; r=4: 4*5/2 = 10 > 8
G(10, 0.5) seed 7                     10   5   r=4: 4*5/2 = 10 <= 10 ; r=5: 5*6/2 = 15 > 10


## C. La méthode de recensement

Pour chaque instance et rang $r$, 40 départs normaux à graines fixes subissent au plus 1 500 itérations d'ascension projetée, avec pas 0,2 et seuil d'arrêt $10^{-7}$. Le gradient riemannien est le gradient projeté sur l'espace tangent de chaque sphère ; les lignes sont renormalisées après chaque pas.

Un départ est un **candidat** si sa norme de gradient est inférieure à $2\times10^{-4}$, sa valeur sous la référence SDP numérique de plus de $10^{-3}\max(1,|p^*|)$ et si aucune des 20 perturbations échantillonnées de rayon 0,02 n'améliore sa valeur. Ces seuils ne certifient ni stationnarité exacte ni maximum local ; le zoom utilise 200 perturbations si un candidat existe. Au rang 1, l'espace tangent de $S^0$ est nul : le test ne peut pas perturber les signes, qui restent ceux du départ.

In [5]:
def bm_f(W, X):
    n = len(W)
    iu = np.triu_indices(n, 1)
    G = X @ X.T
    return 0.5 * np.sum(W[iu] * (1 - G[iu]))

def bm_ascent(W, r, seed, iters=1500, eta=0.2, tol=1e-7):
    n = len(W)
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, r))
    X /= np.linalg.norm(X, axis=1, keepdims=True)
    for t in range(iters):
        grad = -0.5 * (W @ X)
        gT = grad - np.sum(X * grad, axis=1, keepdims=True) * X
        if np.linalg.norm(gT) < tol:
            break
        X = X + eta * gT
        X /= np.linalg.norm(X, axis=1, keepdims=True)
    grad = -0.5 * (W @ X)
    gT = grad - np.sum(X * grad, axis=1, keepdims=True) * X
    return X, bm_f(W, X), float(np.linalg.norm(gT)), t + 1

def est_max_local_empirique(W, X, K=20, eps=0.02, seed=123):
    rng = np.random.default_rng(seed)
    f0 = bm_f(W, X)
    for _ in range(K):
        D = rng.standard_normal(X.shape)
        D -= np.sum(X * D, axis=1, keepdims=True) * X
        nrm = np.linalg.norm(D, axis=1, keepdims=True)
        if np.all(nrm < 1e-12):
            continue  # tangent trivial (r=1 : S^0 est discret), aucune perturbation possible
        D /= nrm
        Xp = X + eps * D
        Xp /= np.linalg.norm(Xp, axis=1, keepdims=True)
        if bm_f(W, Xp) > f0 + 1e-9:
            return False
    return True

print("Outils prets : bm_f (objectif), bm_ascent (gradient projete, renvoie X), est_max_local_empirique (absence d’amélioration échantillonnée)")

Outils prets : bm_f (objectif), bm_ascent (gradient projete, renvoie X), est_max_local_empirique (absence d’amélioration échantillonnée)


In [6]:
# Chaque instance et rang conserve quarante departs et les criteres numeriques du recensement.
# Un candidat combine deficit, petit gradient et perturbations sans amelioration : ce n’est pas un certificat.
STARTS = 40

def recensement(W, rmax):
    n = len(W)
    p_star = sdp_maxcut(W)
    lignes = []
    for r in range(1, rmax + 1):
        fallacieux, converges, vals = 0, 0, []
        for s in range(STARTS):
            X, val, gnorm, _ = bm_ascent(W, r, seed=1000 + s)
            vals.append(val)
            if gnorm < 2e-4:
                converges += 1
                if val < p_star - 1e-3 * max(1.0, abs(p_star)) and est_max_local_empirique(W, X, seed=9000 + s):
                    fallacieux += 1
        lignes.append((r, converges, fallacieux, max(vals), min(vals)))
    return p_star, lignes

print(f"{'instance':32s} {'r':>2s} {'seuil':>6s} {'conv.':>9s} {'candidats':>10s} {'meilleur':>9s} {'pire':>9s}")
RESULTATS = {}
for name, W in INSTANCES:
    n = len(W)
    rs = r_star(n)
    p_star, lignes = recensement(W, rs + 1)
    RESULTATS[name] = (p_star, lignes)
    for r, conv, fal, vmax, vmin in lignes:
        print(f"{name:32s} {r:2d} {'>= r*' if r >= rs else '<  r*':>6s} {conv:5d}/{STARTS:<3d} {fal:10d} {vmax:9.4f} {vmin:9.4f}")

instance                          r  seuil     conv.  candidats  meilleur      pire


C6 (cycle pair, biparti)          1  <  r*    40/40          39    6.0000    2.0000
C6 (cycle pair, biparti)          2  <  r*    40/40           5    6.0000    4.5000
C6 (cycle pair, biparti)          3  <  r*    40/40           0    6.0000    6.0000
C6 (cycle pair, biparti)          4  >= r*    40/40           0    6.0000    6.0000
C6 (cycle pair, biparti)          5  >= r*    40/40           0    6.0000    6.0000


K8 poids uniformes seed 42        1  <  r*    40/40          40   11.1461    5.0249
K8 poids uniformes seed 42        2  <  r*    40/40           0   11.8584   11.8584
K8 poids uniformes seed 42        3  <  r*    40/40           0   11.8584   11.8584
K8 poids uniformes seed 42        4  >= r*    40/40           0   11.8584   11.8584
K8 poids uniformes seed 42        5  >= r*    40/40           0   11.8584   11.8584


G(10, 0.5) seed 7                 1  <  r*    40/40          40   13.0000    5.0000
G(10, 0.5) seed 7                 2  <  r*    40/40          12   14.3249   13.7699
G(10, 0.5) seed 7                 3  <  r*    40/40           0   14.3249   14.3249
G(10, 0.5) seed 7                 4  <  r*    40/40           0   14.3249   14.3249
G(10, 0.5) seed 7                 5  >= r*    40/40           0   14.3249   14.3249
G(10, 0.5) seed 7                 6  >= r*    40/40           0   14.3249   14.3249


**Lecture du recensement.** Les colonnes donnent le nombre de départs à gradient inférieur au seuil, puis les candidats sous la référence SDP numérique pour lesquels aucune amélioration n'a été trouvée parmi 20 perturbations. Ces tests ne prouvent pas un maximum local. Au rang 1, le gradient tangent est nul sur $S^0$ : aucune bascule n'est effectuée et les valeurs sont celles des coupes initiales aléatoires. Les hypothèses de généricité et de régularité du résultat théorique ne sont pas vérifiées ici ; un tableau de zéros ne certifie aucune garantie sur ces matrices.

Sur ce banc Python, le rang 1 compte 39 candidats sur 40 pour le cycle et 40 sur 40 pour les deux autres graphes. Il ne s'agit pas d'une ascension dans l'espace des coupes : le gradient tangent nul laisse les signes initiaux inchangés. Au rang 2, le tableau donne respectivement 5, 0 et 12 candidats ; les rangs suivants n'en donnent aucun dans les départs testés. Le seuil calculé vaut 4, 4 et 5 : l'absence observée commence donc avant lui. Cette observation bornée ne vérifie pas les hypothèses d'un théorème ni l'absence de tout point sous-optimal.


In [7]:
# Contre-exemple positif : exhiber UN maximum local fallacieux concret (G(10,0.5), r=2)
name_g10, W_g10 = INSTANCES[2]
p_star = RESULTATS[name_g10][0]

candidats = []
for s in range(STARTS):
    X, val, gnorm, _ = bm_ascent(W_g10, 2, seed=1000 + s)
    if gnorm < 2e-4 and val < p_star - 1e-3 * max(1.0, abs(p_star)) and est_max_local_empirique(W_g10, X, seed=9000 + s):
        candidats.append((val, s, X))
if candidats:
    val, s_chosen, X_chosen = min(candidats)
    rng = np.random.default_rng(31415)
    ameliorations = []
    for _ in range(200):
        D = rng.standard_normal(X_chosen.shape)
        D -= np.sum(X_chosen * D, axis=1, keepdims=True) * X_chosen
        D /= np.linalg.norm(D, axis=1, keepdims=True)
        Xp = X_chosen + 0.02 * D
        Xp /= np.linalg.norm(Xp, axis=1, keepdims=True)
        ameliorations.append(bm_f(W_g10, Xp) - bm_f(W_g10, X_chosen))
    ameliorations = np.array(ameliorations)
    print(f"Départ seed={1000+s_chosen} (r=2) : valeur {val:.4f}, SDP numérique {p_star:.4f}, déficit {p_star-val:.4f}")
    print(f"Sur 200 perturbations : amélioration maximale {ameliorations.max():.2e}, positives {(ameliorations > 1e-9).sum()}")
    print("Absence d'amélioration échantillonnée, sans certificat de maximum local.")
else:
    X_chosen = None
    print("Aucun candidat au zoom pour ces départs ; aucune autre instance n'est substituée.")


Départ seed=1032 (r=2) : valeur 13.7699, SDP numérique 14.3249, déficit 0.5551
Sur 200 perturbations : amélioration maximale -3.01e-04, positives 0
Absence d'amélioration échantillonnée, sans certificat de maximum local.


**Lecture du zoom.** Le tableau affiche, s'il existe, le candidat retenu et les 200 perturbations essayées. Ne trouver aucune amélioration dans cet échantillon ne prouve pas qu'il n'existe aucun meilleur voisin.

## D. Friction numérique et limites de mesure

La référence CLARABEL est une approximation numérique de la relaxation. La matrice symétrique compte chaque arête deux fois : la somme sur toutes ses entrées porte donc le facteur $1/4$, équivalent au facteur $1/2$ sur $i<j$. Un mauvais facteur changerait la référence et tous les déficits.

La convergence peut être lente près d'un optimum : une limite d'itérations ou un seuil trop strict peut laisser des départs non classés. Le gradient et le déficit sont des seuils numériques ; les perturbations n'établissent ni la signature de la hessienne ni la localité. Au rang 1, $S^0$ a un espace tangent nul : l'ascension ne change aucun signe.

## E. Limites et réserves

- **Taille.** $n \le 10$ : l'énumération exhaustive des coupes ($2^{n-1}$) reste la référence. Rien ne prouve que la phénoménologie (propreté commençant avant $r^*$) persiste à grande échelle.
- **Recensement n'est pas preuve.** Zéro fallacieux observé à $r \ge r^*$ sur 3 instances × 40 départs ne remplace pas le théorème : l'absence observée n'est pas l'absence démontrée. Réciproquement, les résultats théoriques exigent notamment des hypothèses de **généricité et de régularité** non vérifiées ici ; une instance dégénérée pourrait montrer des fallacieux au-dessus du seuil sans le réfuter.
- **Le pont SOS est cité, pas reproduit.** Le résultat SIAM 2023 (SOS univarié sans minima fallacieux) ouvre la lecture « paysage low-rank SOS » de l'EPIC ; ce notebook mesure MaxCut, le pont reste bibliographique.
- **Rounding non traité.** Passer de l'optimum SDP $Y^*$ à une coupe (arrondi hyperplan de Goemans–Williamson) est un autre sujet — ici la cible du paysage est la valeur SDP, pas la coupe.

## Raccord au corpus

**Search-9** (programmation linéaire) → ici la relaxation monte d'un cran (cône semidéfini) puis se défait : convexité exacte contre efficacité non convexe. **Search-4** (recherche locale) : les minima fallacieux sont l'ennemi générique de toute méthode locale. **MGS-15** (Partie 4, série .NET) : la même question de paysage, posée sur des fonctions de test et mesurée par corrélation fitness-distance — l'autre famille, l'autre méthode. Ce notebook est le pilote « calculatoire borné » de l'EPIC #13106 : mesurer sur instances énumérables ce que la théorie garantit au seuil du rang.

## Exercices

Les exercices s'exécutent de bout en bout (cellules stubs valides, rien à décommenter).

### Exercice 1 — Le graphe de Petersen

Construisez $W$ du graphe de Petersen ($n=10$, 3-régulier, cycle extérieur $i\!\!\pmod5$, rayons $i$–$(5+i)$ et étoile intérieure $(5+i)$–$(5+(i+2)\bmod5)$ pour $i<5$), calculez coupe optimale, $p^*$ SDP, puis recensement $r = 1..6$. Le seuil $r^*$ vaut 5 : la propreté commence-t-elle avant, comme sur les trois instances du cours ?

### Exercice 2 — Synchronisation Z2 bruitée

Remplacez $W$ par une instance de synchronisation : $W = s_0 s_0^T + \sigma \cdot N$ avec $s_0 \in \{\pm1\}^{10}$ fixé, $N$ symétrique à diagonale nulle (graine fixée), $\sigma = 0{,}3$. Recensez $r=1..5$ : le bruit crée-t-il des fallacieux au-dessus du seuil ?

### Exercice 3 — Fiabilité du test de max local

Faites varier $\varepsilon \in \{0{,}005, 0{,}02, 0{,}05, 0{,}1\}$ dans `est_max_local_empirique` sur le candidat du zoom, s'il existe : à partir de quel rayon une perturbation trouve-t-elle une amélioration ? Cela explore des perturbations échantillonnées, sans certifier le bassin ni le second ordre.

In [8]:
# Exercice 1 : graphe de Petersen — recensement au seuil r* = 5.
# Indice : W[i,(i+1)%5] pour le cycle exterieur, W[5+i, 5+(i+2)%5] pour l'etoile interieure,
#          W[i, i+5] pour les rayons. Puis reutiliser maxcut_brut, sdp_maxcut, recensement.
def petersen_W():
    # TODO etudiant : construire et retourner la matrice (10, 10) symetrique a diagonale nulle.
    return None  # TODO etudiant

# Exercice 2 : synchronisation Z2 bruitee.
def W_sync_bruitee(sigma=0.3, seed=99):
    # TODO etudiant : s0 uniforme +/-1 fixe par seed, N symetrique diag nulle, retourner W.
    return None  # TODO etudiant

# Exercice 3 : rayon de Bassin du piege.
def rayon_bassin(X, W, eps_list=(0.005, 0.02, 0.05, 0.1), K=100, seed=2718):
    # TODO etudiant : pour chaque eps, retourner la meilleure amelioration trouvee sur K perturbations.
    return None  # TODO etudiant

print("Exercices a completer : petersen_W, W_sync_bruitee, rayon_bassin.")

Exercices a completer : petersen_W, W_sync_bruitee, rayon_bassin.


## À retenir

- La coupe est calculée exhaustivement pour ces trois petits graphes ; CLARABEL donne une référence SDP numérique, avec des écarts visibles dans le tableau.
- La factorisation impose des vecteurs unitaires et rend le paysage non convexe. Le rang 1 est discret : le gradient tangent est nul et les signes initiaux ne changent pas.
- Le recensement distingue déficit, norme du gradient et absence d'amélioration sur 20 perturbations. Le zoom en teste 200 si un candidat existe ; aucun des deux ne certifie un maximum local.
- Les hypothèses de généricité et de régularité des théorèmes de paysage ne sont pas vérifiées pour ces trois matrices : aucune garantie générale n'est certifiée par ce banc.
| Instance | Coupe exhaustive | SDP numérique | Écart numérique | Seuil calculé |
|---|---:|---:|---:|---:|
| Cycle pair C6 | 6,0000 | 6,0000 | voisin de zéro | 4 |
| K8 pondéré, graine 42 | 11,8581 | 11,8584 | 0,0003 | 4 |
| G(10, 0,5), graine 7 | 14,0000 | 14,3249 | 0,3249 | 5 |

Le petit écart sur K8 doit se lire à la précision numérique du solveur ; il ne suffit pas à certifier un écart strict d'intégralité. Le zoom au rang 2 sur G(10, 0,5), graine 1032, atteint 13,7699 contre 14,3249 pour la référence, soit un déficit de 0,5551. Les 200 perturbations ne trouvent aucune amélioration positive ; la meilleure variation est −3,01×10⁻⁴. Ce témoin est une absence d'amélioration dans l'échantillon, sans certification de maximum local.

Le fil rejoint MGS-15 : le rang est un paramètre du paysage étudié, tandis que le recensement reste une expérience finie. Le seuil théorique indique une question à poser sous ses hypothèses, sans transformer ces mesures en garantie de recherche locale.


## Vérifiez votre compréhension

1. Pourquoi comparer coupe exhaustive et SDP ?

<details><summary>Réponse</summary>
La première est une valeur discrète exacte sur ces petites instances ; la seconde est une référence numérique de la relaxation convexe. Sur G(10, 0,5), les valeurs affichées sont 14,0000 et 14,3249 : leur différence est 0,3249.
</details>

2. Que se passe-t-il au rang 1 ?

<details><summary>Réponse</summary>
$S^0$ est discret et son gradient tangent est nul ; les signes initiaux ne changent pas. Les zéros observés aux rangs supérieurs concernent les 40 départs testés, et ne prouvent aucune propriété globale du paysage.
</details>

3. Que prouvent 200 perturbations sans amélioration ?

<details><summary>Réponse</summary>
Elles ne montrent qu'une absence d'amélioration dans cet échantillon, pas un certificat de maximum local.
</details>

## Pour aller plus loin

**Carnet suivant.** [Search-09c — Discrépance combinatoire : colorier ±1 sans déséquilibrer](Search-09c-CombinatorialDiscrepancy.ipynb) — le même geste de « borner un déséquilibre », appliqué cette fois à des sommes de parties, avec une garantie algorithmique.

**Références.** S. Burer, R. D. C. Monteiro, *Local Minima and Convergence in Low-Rank Semidefinite Programming*, Mathematical Programming 103(3), 427–444 (2005) — [DOI 10.1007/s10107-004-0564-1](https://link.springer.com/article/10.1007/s10107-004-0564-1) · G. Blekherman et al. pour le contexte Barvinok–Pataki (1995/1998) · *Low-Rank Univariate Sum of Squares Has No Spurious Local Minima*, SIAM J. Optim. 33(2) (2023) — [DOI 10.1137/22M1516208](https://epubs.siam.org/doi/abs/10.1137/22M1516208) · M. Yurtsever et al., [Scalable Semidefinite Programming](https://web.stanford.edu/~udell/doc/yurtsever21_scalable.pdf) (l'usage pratique de la factorisation). Pilote #13194 / EPIC #13106.

**Carnets voisins.** [Search-4 — Recherche locale](Search-04-LocalSearch.ipynb) · [Search-9 — Programmation linéaire](Search-09-LinearProgramming.ipynb) · [MGS-15 — Analyse de paysage](../Part4-Metaheuristics/MGS-15-LandscapeAnalysis.ipynb) (Partie 4 — l'autre famille, l'autre méthode : la même question de paysage, mesurée par corrélation fitness-distance).
